# Sampling Distributions and the Central Limit Theorem

In the explainer, we saw that every conclusion drawn from a sample carries uncertainty. A poll of 2,000 voters estimates the behaviour of 47 million. A clinical trial of 2,000 patients estimates how a drug will perform across millions. The sample statistic is never exactly right, because we only measured part of the population.

But how wrong could it be? That is the question this notebook answers. We have data from a Phase III clinical trial with 2,000 patients, and we are going to simulate the process of sampling from it. By drawing samples ourselves and watching the results vary, we will build concrete intuition for why sample means bounce around, why they form a predictable shape, and why larger samples produce better estimates.

By the end of this notebook we will be able to:

- Draw random samples from a population using `numpy`
- Build a sampling distribution by computing means from repeated draws
- Explain why the sampling distribution is approximately normal (CLT)
- Describe how sample size affects the spread of sample means
- Calculate and interpret the **standard error** of the mean

In [ ]:
%pip install -q pandas matplotlib seaborn scipy

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
np.random.seed(42)

---

## 1. Loading the population

The explainer made a distinction between the population (the full group we care about) and the sample (the subset we actually have). Normally, we never get to see the population. That is the whole problem.

Here, though, we have an unusual advantage: the full trial dataset of 2,000 patients. Treat it as the population. Having it in front of us means we can compare sample statistics against the known truth, which will make the uncertainty visible in a way that real-world analysis rarely allows.

In [ ]:
patients = pd.read_csv("../data/trial_patients.csv")
outcomes = pd.read_csv("../data/treatment_outcomes.csv")

df = patients.merge(outcomes, on="patient_id")
print(f"Full dataset: {len(df)} patients")
df.head()

Focus on `baseline_score`, the symptom severity score measured before any treatment began. This is the number the trial records at intake, and it will serve as our population for the sampling experiments that follow.

In [ ]:
population = df["baseline_score"].values

pop_mean = population.mean()
pop_std = population.std()

print(f"Population mean: {pop_mean:.2f}")
print(f"Population std:  {pop_std:.2f}")
print(f"Population size: {len(population)}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(population, bins=40, edgecolor="white", alpha=0.7)
ax.axvline(pop_mean, color="red", linewidth=2, label=f"Population mean = {pop_mean:.2f}")
ax.set_xlabel("Baseline score")
ax.set_ylabel("Frequency")
ax.set_title("Distribution of baseline scores (full population)")
ax.legend()
plt.tight_layout()
plt.show()

---

## 2. Drawing a single sample

Imagine we could only recruit 30 patients for this trial instead of 2,000. How close would their mean baseline score be to the true population mean? Run the cell below to find out.

In [ ]:
sample_30 = np.random.choice(population, size=30, replace=False)

print(f"Sample mean:     {sample_30.mean():.2f}")
print(f"Population mean: {pop_mean:.2f}")
print(f"Difference:      {sample_30.mean() - pop_mean:.2f}")

The sample mean lands close to the population mean, but not exactly on it. That gap is **sampling error**, and it is unavoidable whenever we work with a subset. Every sample tells a slightly different story. The question is: how different could that story be?

---

## 3. The sampling distribution of the mean

One sample gave us one mean. But what if we repeated the process thousands of times, each time drawing a fresh batch of 30 patients and recording their mean? The collection of all those sample means forms what statisticians call the **sampling distribution of the mean**.

This is the idea the explainer introduced with the Galton board analogy: each individual ball takes an unpredictable path, but the aggregate pattern is remarkably regular. We are about to see the same thing with data.

In [ ]:
n_draws = 5000
sample_size = 30

sample_means = []
for _ in range(n_draws):
    s = np.random.choice(population, size=sample_size, replace=True)
    sample_means.append(s.mean())

sample_means = np.array(sample_means)

print(f"Mean of sample means:  {sample_means.mean():.2f}")
print(f"Std of sample means:   {sample_means.std():.2f}")
print(f"Population mean:       {pop_mean:.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(sample_means, bins=50, edgecolor="white", alpha=0.7, density=True)
ax.axvline(pop_mean, color="red", linewidth=2, label=f"Population mean = {pop_mean:.2f}")
ax.set_xlabel("Sample mean (n=30)")
ax.set_ylabel("Density")
ax.set_title("Sampling distribution of the mean (5000 draws, n=30)")
ax.legend()
plt.tight_layout()
plt.show()

Two things to notice:

1. **The distribution is centred on the population mean.** On average, samples get the right answer. Some overestimate, some underestimate, but the errors balance out. This is what makes the sample mean an unbiased estimator.
2. **The shape is approximately normal** (bell-curved), even though the population itself may not be perfectly symmetrical.

This is the **Central Limit Theorem** in action. The explainer described it in words; now we can see it. The distribution of sample means tends towards a normal distribution as sample size increases, regardless of the shape of the underlying population.

---

## 4. How sample size affects spread

The explainer noted that larger samples produce tighter estimates, and that the standard error decreases in proportion to the square root of the sample size. We had to take that on trust. Now we can verify it directly by comparing sampling distributions for three different sample sizes.

In [ ]:
sample_sizes = [30, 100, 500]
results = {}

for n in sample_sizes:
    means = [np.random.choice(population, size=n, replace=True).mean() for _ in range(5000)]
    results[n] = np.array(means)
    print(f"n={n:>3d}  |  Mean of means: {results[n].mean():.2f}  |  Std of means: {results[n].std():.2f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)

for ax, n in zip(axes, sample_sizes):
    ax.hist(results[n], bins=40, edgecolor="white", alpha=0.7, density=True)
    ax.axvline(pop_mean, color="red", linewidth=2)
    ax.set_title(f"n = {n}")
    ax.set_xlabel("Sample mean")
    ax.set_xlim(pop_mean - 6, pop_mean + 6)

axes[0].set_ylabel("Density")
fig.suptitle("Sampling distributions for different sample sizes", fontsize=13)
plt.tight_layout()
plt.show()

The pattern is clear. With n=30, the sample means scatter over a wide range. With n=500, they cluster tightly around the truth. This is why the RECOVERY trial enrolled thousands of patients rather than dozens: larger samples produce estimates we can trust more. It is also why a poll of 2,000 people is more reliable than a poll of 200.

---

## 5. Standard error

The **standard error** (SE) of the mean puts a number on the spread we just observed. It is the standard deviation of the sampling distribution, and it tells us how much a sample mean typically deviates from the population mean.

The formula is:

$$SE = \frac{\sigma}{\sqrt{n}}$$

where $\sigma$ is the population standard deviation and $n$ is the sample size. In practice, we replace $\sigma$ with the sample standard deviation $s$, because the population value is usually unknown. The code below compares the theoretical SE to what we actually observed in the simulations.

In [ ]:
for n in sample_sizes:
    theoretical_se = pop_std / np.sqrt(n)
    observed_se = results[n].std()
    print(f"n={n:>3d}  |  Theoretical SE: {theoretical_se:.3f}  |  Observed SE: {observed_se:.3f}")

The theoretical and observed values match closely, which confirms the formula works. Notice the practical implication: doubling the sample size does not halve the standard error. We need to *quadruple* it, because of the square root in the denominator. This is the diminishing-returns relationship the explainer described. Going from 200 to 2,000 respondents produces a dramatic improvement; going from 2,000 to 20,000 produces a much smaller one.

---

## 6. Where does one sample sit?

In real research, we get one sample, not 5,000. We never see the sampling distribution. But knowing it exists, and knowing its shape and spread, is what makes inference possible. It lets us reason about how far our single estimate might be from the truth.

The plot below places our original sample of 30 on the sampling distribution, so we can see where it landed.

In [ ]:
our_sample_mean = sample_30.mean()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(results[30], bins=50, edgecolor="white", alpha=0.7, density=True)
ax.axvline(pop_mean, color="red", linewidth=2, label=f"Population mean = {pop_mean:.2f}")
ax.axvline(our_sample_mean, color="green", linewidth=2, linestyle="--",
           label=f"Our sample mean = {our_sample_mean:.2f}")
ax.set_xlabel("Sample mean (n=30)")
ax.set_ylabel("Density")
ax.set_title("Our sample in context")
ax.legend()
plt.tight_layout()
plt.show()

Our single sample mean is one draw from this distribution. Sometimes it falls near the centre, sometimes further out. The tools of statistical inference exist to quantify that "sometimes" precisely, so we can report not just our estimate but how much trust it deserves.

---

## 7. CLT with a non-normal population

Everything so far used `baseline_score`, which was reasonably well-behaved. But the CLT's real power is that it works even when the underlying population is not normal. To test this, switch to `weight_kg`, which has a different shape with bounded tails.

In [ ]:
weights = df["weight_kg"].values

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Population distribution
axes[0].hist(weights, bins=40, edgecolor="white", alpha=0.7)
axes[0].set_title("Population: weight_kg")
axes[0].set_xlabel("Weight (kg)")
axes[0].set_ylabel("Frequency")

# Sampling distribution of mean, n=30
weight_means = [np.random.choice(weights, size=30, replace=True).mean() for _ in range(5000)]
axes[1].hist(weight_means, bins=50, edgecolor="white", alpha=0.7, density=True)
axes[1].axvline(weights.mean(), color="red", linewidth=2)
axes[1].set_title("Sampling distribution of mean weight (n=30)")
axes[1].set_xlabel("Sample mean weight (kg)")
axes[1].set_ylabel("Density")

plt.tight_layout()
plt.show()

The population has clipped tails (weight is bounded between 45 and 150 kg), yet the sampling distribution of the mean is approximately normal. The CLT holds regardless of the population shape. This is the result that makes inference generalisable: we do not need to know what the population looks like to reason about sample means.

---

## 8. Exercises

The simulations above gave us a feel for how sampling distributions behave. These exercises push that understanding further, asking you to explore different statistics, calculate standard errors by hand, and connect sampling variation to the trial's treatment groups.

### Exercise 1: Sampling distribution of the median

The CLT applies to the mean, but what about other statistics? Build a sampling distribution for the **median** of `baseline_score` using 5000 draws of size 50. Plot it as a histogram and print the standard deviation of the sampling distribution. Is the shape approximately normal?

In [ ]:
# Your code here


### Exercise 2: Standard error by hand

Take a single random sample of 100 patients' `baseline_score` values. Using only that sample (not the population), calculate:

1. The sample mean
2. The sample standard deviation (use `ddof=1`)
3. The estimated standard error ($s / \sqrt{n}$)

Print all three values. Then compare your estimated SE to the theoretical SE using the known population standard deviation.

In [ ]:
# Your code here


### Exercise 3: The effect of sample size on precision

For sample sizes `[10, 25, 50, 100, 250, 500, 1000]`, compute the theoretical standard error ($\sigma / \sqrt{n}$) using the population standard deviation of `baseline_score`. Plot sample size on the x-axis and standard error on the y-axis. What sample size would you need to get the SE below 0.5?

In [ ]:
# Your code here


### Exercise 4: Sampling from treatment vs control

Separately build sampling distributions (5000 draws, n=50) for the mean `week_12_score` of the treatment group and the control group. Plot both distributions on the same histogram (use `alpha=0.5` for transparency). Do the distributions overlap much? What does this tell you about whether the treatment had an effect?

In [ ]:
# Your code here


---

## Summary

In this notebook we:

- Drew random samples from a known population and observed that each sample gives a different mean
- Built **sampling distributions** by repeating the process thousands of times
- Confirmed the **Central Limit Theorem**: the distribution of sample means is approximately normal, regardless of the population shape
- Showed that larger samples produce narrower sampling distributions and more precise estimates
- Calculated the **standard error** and verified it matches $\sigma / \sqrt{n}$
- Placed a single sample mean in context by comparing it to the full sampling distribution

We now have a concrete picture of the uncertainty that the explainer described in the abstract. Every sample mean sits somewhere on a bell curve centred on the truth. The next notebook puts this to practical use: we will run formal **hypothesis tests** to determine whether the treatment in this trial had a statistically significant effect.